# 00 — NumPy for ML

The NumPy you need for machine learning and deep learning: shapes and axes, broadcasting, vectorization, matrix shapes,
randomness and numerical stability. Array basics (creating arrays, simple indexing) are in `01-linear-regression/1-numpy-advance.ipynb`.

## What's in this notebook

- 1\. Why NumPy?
- 2\. Shapes, dtypes and axes
- 3\. Indexing, masks and fancy indexing
- 4\. Views vs copies
- 5\. Broadcasting
- 6\. Vectorization vs Python loops
- 7\. Matrix multiplication and shape rules
- 8\. Randomness and reproducibility
- 9\. Numerical stability
- 10\. Reshape, stack and concatenate
- 11\. Trade-offs
- 12\. Interview questions
- 13\. In my own words

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul", category=RuntimeWarning)

import time
import numpy as np

# 1. WHY NUMPY?

A NumPy array is one block of memory holding values of **one type**. Operations on it run in compiled C code over the whole block,
instead of a Python loop over objects. pandas, scikit-learn and PyTorch tensors are all built on this idea, and their APIs copy NumPy's.

| | Python list | NumPy array |
|---|---|---|
| Stores | pointers to separate Python objects | raw numbers side by side in memory |
| Types | anything, mixed | one dtype (float64, int32, bool, ...) |
| Math on all elements | a loop in Python | one call in C (vectorized) |
| `a + b` | joins the lists | adds element by element |

# 2. SHAPES, DTYPES AND AXES

- `shape` is the size along each dimension: a table of 4 rows and 3 features has shape (4, 3).
- `axis=0` runs **down the rows**, so it gives one result per column; `axis=1` runs **across the columns**, one result per row.
- `keepdims=True` keeps the reduced axis with size 1, which makes the result broadcast back against the original (section 5).
- `float32` takes half the memory of `float64`. Deep learning uses float32 (or smaller); NumPy defaults to float64.

In [2]:
X = np.array([[1.0, 200.0, 3.0],
              [2.0, 180.0, 1.0],
              [3.0, 220.0, 2.0],
              [4.0, 210.0, 5.0]])
print("shape:", X.shape, " ndim:", X.ndim, " dtype:", X.dtype)
print("mean of each column (axis=0):", X.mean(axis=0))
print("mean of each row    (axis=1):", X.mean(axis=1).round(2))
print("axis=1 with keepdims, shape:", X.mean(axis=1, keepdims=True).shape)

big = np.zeros((1_000_000, 10))
print(f"\n1M x 10 array: float64 {big.nbytes / 1e6:.0f} MB, float32 {big.astype(np.float32).nbytes / 1e6:.0f} MB")

shape: (4, 3)  ndim: 2  dtype: float64
mean of each column (axis=0): [  2.5  202.5    2.75]
mean of each row    (axis=1): [68. 61. 75. 73.]
axis=1 with keepdims, shape: (4, 1)

1M x 10 array: float64 80 MB, float32 40 MB


`axis=0` gives 3 numbers (one per feature column), `axis=1` gives 4 (one per row). The same 10 million values take 80 MB as float64 and 40 MB as float32.

# 3. INDEXING, MASKS AND FANCY INDEXING

| Kind | Example | Returns |
|---|---|---|
| slice | `X[1:3, :2]` | a **view** (shares memory with X) |
| boolean mask | `X[X[:, 0] > 2]` | a **copy** of the matching rows |
| fancy (a list of positions) | `X[[0, 3]]` | a **copy** |
| `np.where(cond, a, b)` | `np.where(X > 100, 1, 0)` | a new array, element by element |

In [3]:
print("rows 1-2, first two columns:\n", X[1:3, :2])
print("rows where feature 0 > 2:\n", X[X[:, 0] > 2])
print("rows 0 and 3:\n", X[[0, 3]])
print("np.where(feature 1 > 200, 'high', 'low'):", np.where(X[:, 1] > 200, "high", "low"))

rows 1-2, first two columns:
 [[  2. 180.]
 [  3. 220.]]
rows where feature 0 > 2:
 [[  3. 220.   2.]
 [  4. 210.   5.]]
rows 0 and 3:
 [[  1. 200.   3.]
 [  4. 210.   5.]]
np.where(feature 1 > 200, 'high', 'low'): ['low' 'low' 'high' 'high']


# 4. VIEWS VS COPIES

Slicing doesn't copy data, it gives a new window onto the same memory. Changing the view changes the original.
That makes slicing cheap, and it is a classic source of bugs. Use `.copy()` when you need an independent array.

In [4]:
data = np.arange(10)
window = data[2:5]                 # a view
window[0] = 99
print("after changing the view, the original is:", data)
print("view shares memory:", np.shares_memory(data, window))

safe = data[2:5].copy()
safe[0] = -1
print("after changing a copy, the original is:  ", data, "| shares memory:", np.shares_memory(data, safe))

after changing the view, the original is: [ 0  1 99  3  4  5  6  7  8  9]
view shares memory: True
after changing a copy, the original is:   [ 0  1 99  3  4  5  6  7  8  9] | shares memory: False


Writing 99 into the view changed `data[2]`. The copy is independent. (pandas has the same issue, covered in the pandas notebook.)

# 5. BROADCASTING

Broadcasting lets arrays of different shapes work together without copying data. Compare the shapes **from the right**:
two dimensions fit if they are equal or one of them is 1, and the 1 is stretched to match.

```text
X            (4, 3)
X.mean(0)       (3,)   ->  treated as (1, 3), stretched to (4, 3)      OK: standardize each column
X.mean(1)       (4,)   ->  (1, 4) against (4, 3): 4 vs 3                ERROR
X.mean(1, keepdims=True)  (4, 1)  ->  stretched to (4, 3)                OK: centre each row
```

Two places it is used all the time:
- **standardizing** features: `(X − X.mean(axis=0)) / X.std(axis=0)`
- **all pairwise distances** without a loop: (n, 1, d) − (1, m, d) gives (n, m, d), then sum over the last axis

In [5]:
Z = (X - X.mean(axis=0)) / X.std(axis=0)
print("standardized columns: mean", Z.mean(axis=0).round(6), " std", Z.std(axis=0).round(6))

try:
    X - X.mean(axis=1)
except ValueError as e:
    print("\nX - X.mean(axis=1) fails:", e)
print("X - X.mean(axis=1, keepdims=True) works, shape:", (X - X.mean(axis=1, keepdims=True)).shape)

rng = np.random.default_rng(0)
A, B = rng.normal(size=(5, 3)), rng.normal(size=(4, 3))
dist = np.sqrt(((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=-1))      # (5, 1, 3) - (1, 4, 3) -> (5, 4, 3) -> (5, 4)
print("\npairwise distance matrix shape:", dist.shape)

standardized columns: mean [0. 0. 0.]  std [1. 1. 1.]

X - X.mean(axis=1) fails: operands could not be broadcast together with shapes (4,3) (4,) 
X - X.mean(axis=1, keepdims=True) works, shape: (4, 3)

pairwise distance matrix shape: (5, 4)


After standardizing, every column has mean 0 and std 1. Subtracting the row means fails until `keepdims=True` keeps their shape as (4, 1).
The distance matrix between 5 and 4 points comes out as (5, 4) with no Python loop.

The catch: broadcasting builds the full (n, m, d) intermediate array. For 10,000 × 10,000 points with 100 features that is 10 billion numbers (80 GB),
so large distance computations use the identity ‖a − b‖² = ‖a‖² + ‖b‖² − 2·a·b with a matrix multiply, or process in chunks.

# 6. VECTORIZATION VS PYTHON LOOPS

"Vectorized" means one NumPy call over the whole array instead of a Python loop over its elements.

In [6]:
x = rng.normal(size=1_000_000)
w = rng.normal(size=1_000_000)

start = time.perf_counter()
total = 0.0
for i in range(len(x)):
    total += x[i] * w[i]
loop_s = time.perf_counter() - start

start = time.perf_counter()
fast = x @ w
vec_s = time.perf_counter() - start

print(f"Python loop: {loop_s * 1000:.0f} ms   vectorized: {vec_s * 1000:.2f} ms   -> about {loop_s / vec_s:,.0f}x faster")
print("same answer:", np.isclose(total, fast))

Python loop: 182 ms   vectorized: 1.04 ms   -> about 175x faster
same answer: True


The same dot product of two 1-million-element vectors takes 182 ms in a Python loop and 1.04 ms vectorized, about 175x faster in this run
(the exact ratio varies by machine). Loops in ML code are a performance bug unless the algorithm is truly sequential.

# 7. MATRIX MULTIPLICATION AND SHAPE RULES

- `A * B` multiplies **element by element** (with broadcasting).
- `A @ B` is **matrix multiplication**: (n, d) @ (d, k) → (n, k). The inner sizes must match.
- A linear layer, logistic regression and the first step of a neural network are all `X @ W + b`.
- `np.einsum` writes any of these with index letters, for example `"nd,dk->nk"`; transformer code uses it a lot.

In [7]:
X_batch = rng.normal(size=(32, 10))       # 32 rows, 10 features
W = rng.normal(size=(10, 4))              # 10 inputs -> 4 outputs
b = np.zeros(4)

out = X_batch @ W + b
print("(32, 10) @ (10, 4) + (4,) ->", out.shape)
print("einsum gives the same result:", np.allclose(out, np.einsum("nd,dk->nk", X_batch, W) + b))
try:
    X_batch @ W.T
except ValueError as e:
    print("(32, 10) @ (4, 10) fails:", str(e)[:70], "...")

(32, 10) @ (10, 4) + (4,) -> (32, 4)
einsum gives the same result: True
(32, 10) @ (4, 10) fails: matmul: Input operand 1 has a mismatch in its core dimension 0, with g ...


# 8. RANDOMNESS AND REPRODUCIBILITY

Use a **Generator**: `rng = np.random.default_rng(seed)`, then `rng.normal(...)`, `rng.permutation(...)`.
The old `np.random.seed()` sets one global state that every library shares, so code elsewhere can change your sequence.
A seed makes results repeatable; it doesn't make them better. Report results over several seeds when they vary.

In [8]:
first = np.random.default_rng(42).permutation(10)
again = np.random.default_rng(42).permutation(10)
print("same seed, same shuffle:", first, np.array_equal(first, again))

n = 10
order = np.random.default_rng(0).permutation(n)          # a train/test split by hand
train_idx, test_idx = order[:8], order[8:]
print("train rows:", np.sort(train_idx), " test rows:", np.sort(test_idx))

same seed, same shuffle: [5 6 0 7 3 2 4 9 1 8] True
train rows: [0 2 3 4 5 6 7 9]  test rows: [1 8]


# 9. NUMERICAL STABILITY

Floating point has limits: `exp(1000)` overflows to infinity, and `0.1 + 0.2` is not exactly `0.3`.

- **Softmax:** subtract the largest value first. The result is mathematically the same, and no `exp` can overflow.
- **Log-sum-exp:** `log Σ exp(zᵢ) = m + log Σ exp(zᵢ − m)` with m = max(z). Used inside cross-entropy, so frameworks take raw logits.
- **Comparing floats:** use `np.isclose`, never `==`.

In [9]:
logits = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(logits) / np.exp(logits).sum()
stable = np.exp(logits - logits.max()) / np.exp(logits - logits.max()).sum()
print("naive softmax: ", naive)
print("stable softmax:", stable.round(4))

m = logits.max()
print("log-sum-exp, stable:", round(m + np.log(np.exp(logits - m).sum()), 4))
print("\n0.1 + 0.2 == 0.3:", 0.1 + 0.2 == 0.3, "  np.isclose:", np.isclose(0.1 + 0.2, 0.3))

naive softmax:  [nan nan nan]
stable softmax: [0.09   0.2447 0.6652]
log-sum-exp, stable: 1002.4076

0.1 + 0.2 == 0.3: False   np.isclose: True


The naive softmax returns `nan` for every class because `exp(1000)` is infinite; subtracting the maximum gives the right answer, [0.09, 0.2447, 0.6652].

# 10. RESHAPE, STACK AND CONCATENATE

- `reshape` changes the shape without copying when it can; `-1` means "work out this size".
- `x[:, None]` (or `np.newaxis`) adds an axis of size 1, as used for broadcasting in section 5.
- `np.concatenate` joins along an **existing** axis; `np.stack` creates a **new** axis.
- Images are often flattened for classical models: (n, 28, 28) → (n, 784).

In [10]:
images = rng.random((100, 28, 28))
flat = images.reshape(len(images), -1)
print("images", images.shape, "-> flattened", flat.shape)

a, b = np.ones((2, 3)), np.zeros((2, 3))
print("concatenate on axis 0:", np.concatenate([a, b], axis=0).shape, "  stack:", np.stack([a, b]).shape)
print("1-D (3,) -> column (3, 1) with [:, None]:", np.arange(3)[:, None].shape)

images (100, 28, 28) -> flattened (100, 784)
concatenate on axis 0: (4, 3)   stack: (2, 2, 3)
1-D (3,) -> column (3, 1) with [:, None]: (3, 1)


# 11. TRADE-OFFS

| Decision | Options | What you gain and what you pay | Usual choice |
|---|---|---|---|
| Precision | float64 / float32 / float16, bfloat16 | float64: most accurate, double the memory and slower on GPUs. float32: the deep-learning standard. 16-bit: half again, needs care with overflow | float64 for classical ML and statistics, float32 or lower for deep learning |
| Loops vs vectorized | Python loop / NumPy call | loops are easy to write but 100–1000x slower; vectorized code is fast but can create huge temporary arrays (section 5) | vectorize; chunk the data when the intermediate arrays get too big |
| Views vs copies | slice / `.copy()` | views are free but share memory, so writes leak; copies are safe but cost memory and time | slice for reading, copy before writing |
| NumPy vs pandas vs PyTorch | — | NumPy: fast numeric arrays, no labels. pandas: labelled, mixed-type tables with joins and group-by, slower. PyTorch: like NumPy plus GPUs and automatic gradients | pandas to prepare tabular data, NumPy inside algorithms, PyTorch for deep learning |

# 12. INTERVIEW QUESTIONS

### Q1. Why is NumPy faster than Python lists?
An array stores numbers of one type side by side in memory, and operations run in compiled C over the whole block (vectorized),
with no per-element Python overhead. A list stores pointers to separate Python objects and needs a Python loop.

### Q2. What does `axis=0` mean?
The operation runs down the rows, so the result has one value per column. `X.mean(axis=0)` gives the mean of each feature; `axis=1` gives one value per row.

### Q3. Explain broadcasting.
Shapes are compared from the right; two sizes fit if they are equal or one of them is 1, which is stretched without copying data.
(4, 3) minus (3,) works and centres each column; (4, 3) minus (4,) fails unless you reshape it to (4, 1).

### Q4. View or copy: what does slicing return, and why does it matter?
A basic slice returns a view that shares memory, so writing to it changes the original. Boolean and fancy indexing return copies.
Call `.copy()` before modifying a slice you want to keep separate.

### Q5. `*` vs `@`?
`*` multiplies element by element; `@` is matrix multiplication, where (n, d) @ (d, k) gives (n, k) and the inner sizes must match.

### Q6. How do you compute all pairwise distances without a loop, and what is the risk?
Broadcast (n, 1, d) − (1, m, d) and sum the squares over the last axis. The intermediate is n·m·d numbers, which can exhaust memory,
so for large inputs use ‖a‖² + ‖b‖² − 2·A·Bᵀ with one matrix multiply, or work in chunks.

### Q7. Why subtract the maximum in softmax?
`exp` of a large logit overflows to infinity and the result becomes `nan`. Subtracting the maximum doesn't change the softmax but keeps every exponent ≤ 0.

### Q8. Why `default_rng` instead of `np.random.seed`?
A Generator object keeps its own state, so other code can't change your random sequence, and you can pass separate generators to separate components.
`np.random.seed` sets one global state shared by everything.

### Q9. float32 vs float64?
float32 uses half the memory and is much faster on GPUs, with about 7 significant digits; float64 has about 16. Deep learning trains in float32 or lower;
statistics and classical ML usually keep float64.

### Q10. `reshape` vs `concatenate` vs `stack`?
`reshape` changes the shape of one array (often without copying). `concatenate` joins arrays along an existing axis; `stack` adds a new axis.

### Q11. How would you speed up slow NumPy code?
Remove Python loops (vectorize), avoid repeated copies and huge temporaries, use float32 if precision allows, use matrix multiplications
(they call optimized BLAS libraries), and process large data in chunks.

# 13. IN MY OWN WORDS

Write this from memory, without looking above:

- axis=0 vs axis=1:
- The broadcasting rule, with one example that works and one that fails:
- When slicing changes my original array:
- Why softmax subtracts the maximum: